In [1]:
from dataclasses import dataclass, field
from typing import Optional
import math



In [2]:
class ProtocolState:
    """Trạng thái toàn bộ protocol tại một thời điểm"""
    sol_price: float          # SOL/USD (từ oracle)
    sol_in_reserve: float     # Tổng SOL trong collateral pool
    hyusd_supply: float       # Tổng hyUSD đang lưu hành
    xsol_supply: float        # Tổng xSOL đang lưu hành
    stability_pool: float     # hyUSD đang stake trong Stability Pool
    xsol_in_pool: float = 0   # xSOL trong Stability Pool (sau drawdown)
 
    # Fees (basis points, 1 bp = 0.01%)
    hyusd_mint_fee_bp: int = 10      # 0.10%
    hyusd_redeem_fee_bp: int = 10
    xsol_mint_fee_bp: int = 10
    xsol_redeem_fee_bp: int = 10
 
 

class NAVResult:
    hyusd_nav_sol: float   # 1 hyUSD = bao nhiêu SOL
    xsol_nav_sol: float    # 1 xSOL = bao nhiêu SOL
    xsol_nav_usd: float    # 1 xSOL = bao nhiêu USD
    fixed_reserve: float   # SOL dành để back hyUSD
    variable_reserve: float # SOL dành cho xSOL
 
 

class HealthMetrics:
    cr: float              # Collateral Ratio (%)
    el: float              # Effective Leverage
    mode: str              # "HEALTHY" | "MODE_1" | "MODE_2" | "CRITICAL"
    xsol_mcap_usd: float
    tvl_usd: float
 
 
class HarvestResult:
    yield_sol: float       # SOL yield trong epoch
    yield_usd: float       # USD value của yield
    hyusd_minted: float    # hyUSD mới được mint
    new_share_price: float # Share price mới của Stability Pool
    apy_estimate: float    # APY ước tính
 
 
class DrawdownResult:
    hyusd_burned: float    # hyUSD bị burn từ pool
    xsol_minted: float     # xSOL được mint thay thế
    cr_before: float
    cr_after: float
    cr_improvement: float

In [3]:
class HyloProtocol:
    CR_HEALTHY = 150.0
    CR_MODE1   = 150.0
    CR_MODE2   = 130.0
    CR_CRITICAL = 100.0

    def __init__(self, state: ProtocolState):
        self.state = state
        
    def calc_collateral_tvl(self) -> float:
        return self.state.sol_price * self.state.sol_in_reserve
    
    def calc_xsol_price(self) -> float:
        return (calc_collateral_tvl(self) - self.state.hyusd_supply) / self.state.xsol_supply

    def calc_effective_leverage(self) -> float:
        return calc_collateral_tvl(self) / (self.state.xsol_supply * calc_xsol_price(self))

    def calc_hyusd_nav_sol(self) -> float:
        """
        hyUSD NAV (SOL) = $1 / SOL price
        Bao nhiêu SOL cần để back 1 hyUSD
        """
        return 1.0 / self.state.sol_price

    def calc_nav(self) -> NAVResult:
        """
        Tính toàn bộ NAV cho hyUSD và xSOL
        """
        hyusd_nav_sol = self.calc_hyusd_nav_sol()

        # Fixed Reserve = bao nhiêu SOL cần để back toàn bộ hyUSD
        fixed_reserve = hyusd_nav_sol * self.state.hyusd_supply

        # Variable Reserve = phần còn lại thuộc về xSOL
        variable_reserve = self.state.sol_in_reserve - fixed_reserve

        # xSOL NAV
        if self.state.xsol_supply > 0 and variable_reserve > 0:
            xsol_nav_sol = variable_reserve / self.state.xsol_supply
        else:
            xsol_nav_sol = 0.0

        xsol_nav_usd = xsol_nav_sol * self.state.sol_price

        return NAVResult(
            hyusd_nav_sol=hyusd_nav_sol,
            xsol_nav_sol=xsol_nav_sol,
            xsol_nav_usd=xsol_nav_usd,
            fixed_reserve=fixed_reserve,
            variable_reserve=variable_reserve,
        )


    def calc_health(self) -> HealthMetrics:
    
        nav = self.calc_nav()

        # CR = Total SOL / Fixed Reserve
        if nav.fixed_reserve > 0:
            cr = (self.state.sol_in_reserve / nav.fixed_reserve) * 100
        else:
            cr = float('inf')

        # EL = Total SOL / Variable Reserve
        if nav.variable_reserve > 0:
            el = self.state.sol_in_reserve / nav.variable_reserve
        else:
            el = float('inf')

        # xSOL Market Cap
        xsol_mcap = nav.xsol_nav_usd * self.state.xsol_supply

        # TVL
        tvl = self.state.sol_in_reserve * self.state.sol_price

        # Determine mode
        if cr >= self.CR_HEALTHY:
            mode = "HEALTHY"
        elif cr >= self.CR_MODE2:
            mode = "MODE 1"
        elif cr > self.CR_CRITICAL:
            mode = "MODE 2"
        else:
            mode = "CRITICAL"

        return HealthMetrics(
            cr=cr,
            el=el,
            mode=mode,
            xsol_mcap_usd=xsol_mcap,
            tvl_usd=tvl,
        )

    def adjust_fees_for_mode(self) -> dict:

        health = self.calc_health()
        cr = health.cr

        if cr >= 150:
            # Healthy — fees bình thường
            fees = {
                "hyusd_mint":   10,   # 0.10%
                "hyusd_redeem": 10,
                "xsol_mint":    10,
                "xsol_redeem":  10,
                "mode": "HEALTHY"
            }
        elif cr >= 130:
            # Mode 1 — fee controls
            severity = (150 - cr) / 20   # 0 → 1 khi CR đi từ 150 → 130
            fees = {
                "hyusd_mint":   int(10 + severity * 90),   # 10 → 100 bp
                "hyusd_redeem": int(10 - severity * 10),   # 10 → 0 bp
                "xsol_mint":    int(10 - severity * 10),   # 10 → 0 bp
                "xsol_redeem":  int(10 + severity * 90),   # 10 → 100 bp
                "mode": "MODE_1"
            }
        else:
            # Mode 2 — aggressive
            severity = min((130 - cr) / 30, 1.0)
            fees = {
                "hyusd_mint":   200,   # 2.00% — rất đắt
                "hyusd_redeem": 0,     # free
                "xsol_mint":    0,     # free
                "xsol_redeem":  200,   # 2.00% — rất đắt
                "mode": "MODE_2"
            }

        # Update state
        self.state.hyusd_mint_fee_bp = fees["hyusd_mint"]
        self.state.hyusd_redeem_fee_bp = fees["hyusd_redeem"]
        self.state.xsol_mint_fee_bp = fees["xsol_mint"]
        self.state.xsol_redeem_fee_bp = fees["xsol_redeem"]

        return fees

    def mint_hyusd(self, sol_deposit: float) -> dict:
        """User deposit SOL → nhận hyUSD"""
        fee = self.state.hyusd_mint_fee_bp / 10000
        hyusd_out = sol_deposit * self.state.sol_price * (1 - fee)
        self.state.sol_in_reserve += sol_deposit
        self.state.hyusd_supply += hyusd_out
        return {"sol_deposited": sol_deposit, "hyusd_received": hyusd_out, "fee_pct": fee * 100}

    def redeem_hyusd(self, hyusd_amount: float) -> dict:
        """User burn hyUSD → nhận SOL"""
        fee = self.state.hyusd_redeem_fee_bp / 10000
        sol_out = (hyusd_amount / self.state.sol_price) * (1 - fee)
        self.state.hyusd_supply -= hyusd_amount
        self.state.sol_in_reserve -= sol_out
        return {"hyusd_burned": hyusd_amount, "sol_received": sol_out, "fee_pct": fee * 100}

    def mint_xsol(self, sol_deposit: float) -> dict:
        """User deposit SOL → nhận xSOL"""
        nav = self.calc_nav()
        fee = self.state.xsol_mint_fee_bp / 10000
        if nav.xsol_nav_sol > 0:
            xsol_out = (sol_deposit * (1 - fee)) / nav.xsol_nav_sol
        else:
            xsol_out = 0
        self.state.sol_in_reserve += sol_deposit
        self.state.xsol_supply += xsol_out
        return {"sol_deposited": sol_deposit, "xsol_received": xsol_out, "fee_pct": fee * 100}

    def redeem_xsol(self, xsol_amount: float) -> dict:
        """User burn xSOL → nhận SOL"""
        nav = self.calc_nav()
        fee = self.state.xsol_redeem_fee_bp / 10000
        sol_out = xsol_amount * nav.xsol_nav_sol * (1 - fee)
        self.state.xsol_supply -= xsol_amount
        self.state.sol_in_reserve -= sol_out
        return {"xsol_burned": xsol_amount, "sol_received": sol_out, "fee_pct": fee * 100}
    def harvest_yield(self, lst_apy: float, days_elapsed: float = 3.0) -> Optional[HarvestResult]:
        """
        Harvest LST yield vào Stability Pool
        lst_apy: APY của LST (ví dụ 0.07 = 7%)
        days_elapsed: số ngày từ epoch trước
        """
        if self.state.stability_pool <= 0:
            return None

        # Yield sinh ra trong epoch
        epoch_yield_rate = lst_apy * (days_elapsed / 365)
        yield_sol = self.state.sol_in_reserve * epoch_yield_rate
        yield_usd = yield_sol * self.state.sol_price

        # Mint hyUSD mới tương ứng với yield (backed bởi excess collateral)
        hyusd_minted = yield_usd

        # Inject vào Stability Pool
        old_pool = self.state.stability_pool
        self.state.sol_in_reserve += yield_sol
        self.state.hyusd_supply += hyusd_minted
        self.state.stability_pool += hyusd_minted

        # Share price sau harvest (giả sử shares không đổi)
        # new_share_price = new_pool / old_pool
        new_share_price = self.state.stability_pool / old_pool if old_pool > 0 else 1.0

        # APY estimate
        apy = (new_share_price - 1) * (365 / days_elapsed)

        # Tính APY thực với yield concentration
        pool_ratio = self.state.stability_pool / (self.state.hyusd_supply or 1)
        amplified_apy = lst_apy / pool_ratio if pool_ratio > 0 else 0

        return HarvestResult(
            yield_sol=yield_sol,
            yield_usd=yield_usd,
            hyusd_minted=hyusd_minted,
            new_share_price=new_share_price,
            apy_estimate=amplified_apy,
        )

    def stability_pool_drawdown(self, target_cr: float = 140.0) -> Optional[DrawdownResult]:
        """
        Kích hoạt Stability Pool drawdown khi CR < 130%
        Burn hyUSD từ pool → Mint xSOL thay thế
        target_cr: CR mục tiêu sau drawdown
        """
        health = self.calc_health()
        nav = self.calc_nav()

        if health.cr >= self.CR_MODE2:
            return None  # Không cần drawdown

        cr_before = health.cr

        # Tính lượng hyUSD cần burn để đạt target_cr
        # CR = sol_in_reserve / (hyusd_nav_sol * hyusd_supply)
        # target_cr/100 = sol_in_reserve / (hyusd_nav_sol * new_hyusd_supply)
        # new_hyusd_supply = sol_in_reserve / (hyusd_nav_sol * target_cr/100)
        target_hyusd = self.state.sol_in_reserve / (nav.hyusd_nav_sol * target_cr / 100)
        hyusd_to_burn = self.state.hyusd_supply - target_hyusd

        # Không thể burn nhiều hơn pool size
        hyusd_to_burn = min(hyusd_to_burn, self.state.stability_pool)
        hyusd_to_burn = max(hyusd_to_burn, 0)

        if hyusd_to_burn <= 0:
            return None

        # SOL được "giải phóng" từ Fixed Reserve
        sol_freed = hyusd_to_burn * nav.hyusd_nav_sol

        # Mint xSOL tương ứng với giá xSOL hiện tại
        if nav.xsol_nav_sol > 0:
            xsol_to_mint = sol_freed / nav.xsol_nav_sol
        else:
            # xSOL price = 0, dùng giá mới sau khi variable reserve tăng
            new_variable = nav.variable_reserve + sol_freed
            xsol_to_mint = new_variable / (self.state.xsol_supply + 1) if self.state.xsol_supply > 0 else 1

        # Cập nhật state
        self.state.hyusd_supply -= hyusd_to_burn
        self.state.stability_pool -= hyusd_to_burn
        self.state.xsol_supply += xsol_to_mint
        self.state.xsol_in_pool += xsol_to_mint  # xSOL ghi nhận cho pool holders

        # CR mới
        health_after = self.calc_health()

        return DrawdownResult(
            hyusd_burned=hyusd_to_burn,
            xsol_minted=xsol_to_mint,
            cr_before=cr_before,
            cr_after=health_after.cr,
            cr_improvement=health_after.cr - cr_before,
        )  

    def print_state(self, label: str = ""):
        nav = self.calc_nav()
        health = self.calc_health()
        sep = "─" * 55

        print(f"\n{'=' * 55}")
        if label:
            print(f"  {label}")
        print(sep)
        print(f"  SOL Price          : ${self.state.sol_price:>12,.2f}")
        print(f"  SOL in Reserve     : {self.state.sol_in_reserve:>12,.2f} SOL")
        print(f"  TVL                : ${health.tvl_usd:>12,.2f}")
        print(sep)
        print(f"  hyUSD Supply       : {self.state.hyusd_supply:>12,.2f}")
        print(f"  hyUSD NAV (SOL)    : {nav.hyusd_nav_sol:>12.6f} SOL")
        print(f"  Fixed Reserve      : {nav.fixed_reserve:>12,.2f} SOL")
        print(sep)
        print(f"  xSOL Supply        : {self.state.xsol_supply:>12,.4f}")
        print(f"  xSOL NAV (SOL)     : {nav.xsol_nav_sol:>12.6f} SOL")
        print(f"  xSOL NAV (USD)     : ${nav.xsol_nav_usd:>12,.2f}")
        print(f"  Variable Reserve   : {nav.variable_reserve:>12,.2f} SOL")
        print(sep)
        print(f"  Collateral Ratio   : {health.cr:>12.2f}%   {health.mode}")
        el_str = f"{health.el:.2f}×" if health.el != float('inf') else "∞"
        print(f"  Effective Leverage : {el_str:>12}")
        print(sep)
        print(f"  Stability Pool     : {self.state.stability_pool:>12,.2f} hyUSD")
        if self.state.xsol_in_pool > 0:
            print(f"  xSOL in Pool       : {self.state.xsol_in_pool:>12.4f} xSOL")
        print(f"  Fees (mint/redeem) : hyUSD {self.state.hyusd_mint_fee_bp/100:.2f}%/{self.state.hyusd_redeem_fee_bp/100:.2f}%"
              f"  xSOL {self.state.xsol_mint_fee_bp/100:.2f}%/{self.state.xsol_redeem_fee_bp/100:.2f}%")
        print(f"{'=' * 55}")
        

    